# 08. Design, training and model selection

**The question this notebook answers:** how was the model built on top of the contract, and why is the model we ship the one we ship, rather than one we liked the look of after seeing the test results?

**What the brief asks (Part B):**
1. **Design first:** commit a one-page `DESIGN.md` before any model code (framing, time split, metrics, baseline, constraints, grade and interest rate as inputs or not), and update it at the end.
2. **One command trains the model from the contract:** `transform.joblib` unchanged as the first step, inputs validated against `data_spec.json` (fail fast), settings in a config file, fixed seeds, every experiment logged, **nothing from `loan_servicing.csv`**, and metadata saved with the model (git commit, training-data hash, training date, contract version, thresholds).
3. **Models:** a **scorecard** (logistic regression on the contract's WoE values); **one stronger model** with justified **monotone constraints**; **class imbalance** handled; **time-aware tuning**; **calibrated** probabilities; the **ablation** raw fields vs factor scores vs both.

The work itself is done by `uv run python -m credit_risk.model.train --config config/train.yaml` (about a minute). This notebook does not retrain or save anything: it reads what that run recorded (`models/metadata.json`, `reports/experiments.csv`) and re-checks the key numbers live on the same time folds.

**Terms used in this notebook**
- **PD (probability of default):** the model's estimate, between 0 and 1, that a loan will end in default.
- **AUC:** the chance that a randomly picked defaulted loan gets a higher PD than a randomly picked repaid loan. 0.5 is guessing, 1.0 is perfect; credit models on application data typically land around 0.65 to 0.75.
- **Brier score:** the average squared gap between the PD and what happened (0 or 1). Lower is better; it rewards honest probabilities, not just a good ranking.
- **Time fold:** train on earlier loans, test on later ones, as the model will be used. Two folds here: 2013 → 2014 H1, and up to 2014 H1 → 2014 H2.
- **Scorecard:** a logistic regression whose inputs are WoE values, so each field adds a fixed number of points per bin and the whole model fits on one page.
- **Gradient boosting (LightGBM):** many small decision trees, each correcting the errors of the ones before. More flexible than a scorecard, harder to explain.
- **Monotone constraint:** a rule that the model's PD may only move one way as a field rises (for example, never fall as debt-to-income rises).
- **Calibration:** adjusting scores so that, among loans given a PD of 10%, about 10% actually default.

In [1]:
import os, json, subprocess, textwrap, warnings
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from credit_risk.io import load_applications
from credit_risk.model.data import SERVICING_COLUMNS, build_frame, data_hash, load_config, split
from credit_risk.model.inputs import CONSTRAINTS, INPUT_SETS, PLATFORM_COLUMNS, SCORECARD_COLUMNS
from credit_risk.model.models import LgbmPD, LogitPD, LogOddsAverage
from credit_risk.model.train import AVERAGE_MIN_GAIN, MIN_GAIN_FOR_BOOSTING, MIN_GAIN_TO_ADD_FACTORS, Trainer, boosted_model_wins
from credit_risk.spec import ContractViolation

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)

cfg = load_config("config/train.yaml")
meta = json.load(open("models/metadata.json"))
sel = meta["selection"]
log = pd.read_csv("reports/experiments.csv")
run = log[log.run_id == meta["run_id"]]
tr = Trainer(cfg)   # loads the contract, builds the modelling frame and the time folds; it fits and writes nothing until run() is called
print(f"saved model: {meta['model']} (training run {meta['run_id']}, {len(run)} trials logged in that run, {len(log):,} in the whole log)")

saved model: scorecard_boosted_average (training run 20261102-080903, 221 trials logged in that run, 1,139 in the whole log)


## 1. Design came first

The brief asks for `DESIGN.md` to be committed **before any model code**. The git history shows the order:

In [2]:
def first_commit(path):
    out = subprocess.check_output(["git", "log", "--diff-filter=A", "--format=%h %ad %s", "--date=format:%d %b %H:%M", "--", path], text=True).strip().splitlines()
    return out[-1] if out else "not found"
for p in ["DESIGN.md", "src/credit_risk/model/train.py", "src/credit_risk/model/models.py", "src/credit_risk/model/predict.py"]:
    print(f"{p:<38} first committed: {first_commit(p)}")
print("\nsections of DESIGN.md:")
print("\n".join("  " + l for l in Path("DESIGN.md").read_text().splitlines() if l.startswith("## ")))

DESIGN.md                              first committed: 7505121 31 Oct 12:29 docs: add the design for the PD model (split, inputs, models, metrics, policy, lineage, serving)
src/credit_risk/model/train.py         first committed: 1a441cf 31 Oct 12:51 feat: train and evaluate the PD models (grade baseline, WoE scorecard, monotone LightGBM), add inference with reason codes, validation and tests
src/credit_risk/model/models.py        first committed: 1a441cf 31 Oct 12:51 feat: train and evaluate the PD models (grade baseline, WoE scorecard, monotone LightGBM), add inference with reason codes, validation and tests
src/credit_risk/model/predict.py       first committed: 1a441cf 31 Oct 12:51 feat: train and evaluate the PD models (grade baseline, WoE scorecard, monotone LightGBM), add inference with reason codes, validation and tests

sections of DESIGN.md:
  ## 1. Problem
  ## 2. Data and time split
  ## 3. Inputs and the platform's own score (revised, see section 11)
  ## 4. Models
  ## 5.

**Reading it.** `DESIGN.md` went in first, on its own, 22 minutes before the first model code. Sections 1 to 9 are the plan (framing, split, inputs and the platform's score, models, metrics, policy, lineage, constraints, production). Sections 10 to 12 were added later and record what changed and why, so a reader can compare intention with outcome.

## 2. Splitting in time around the fit cutoff

Everything the model learns comes from loans issued up to 31 December 2014. Inside that window, two **time folds** are used to tune and choose; 2015 is only for the final report; 2016 is scored without labels.

In [3]:
f = tr.frame
rows = []
for i, (a, b) in enumerate(tr.folds, 1):
    rows.append({"window": f"fold {i}: train", "issued": f"{a.issue_d.min():%b %Y} to {a.issue_d.max():%b %Y}", "loans": len(a), "default_rate": a.defaulted.mean(), "used for": "tuning and choosing"})
    rows.append({"window": f"fold {i}: validate", "issued": f"{b.issue_d.min():%b %Y} to {b.issue_d.max():%b %Y}", "loans": len(b), "default_rate": b.defaulted.mean(), "used for": "tuning and choosing"})
rows.append({"window": "final training", "issued": f"{tr.train.issue_d.min():%b %Y} to {tr.train.issue_d.max():%b %Y}", "loans": len(tr.train), "default_rate": tr.train.defaulted.mean(), "used for": "fitting the saved models"})
for name, w, use in [("2015 H1", cfg["policy_window"], "checking the decision thresholds (notebook 09)"), ("2015 H2", cfg["report_window"], "reported results (notebook 09)")]:
    s = split(f, w["start"], w["end"])
    rows.append({"window": name, "issued": f"{s.issue_d.min():%b %Y} to {s.issue_d.max():%b %Y}", "loans": len(s), "default_rate": s.defaulted.mean(), "used for": use})
hold = pd.read_csv("outputs/holdout_scores.csv")
rows.append({"window": "2016 holdout", "issued": "2016", "loans": len(hold), "default_rate": np.nan, "used for": "scored only, no labels (notebook 10)"})
print(pd.DataFrame(rows).round(3).to_string(index=False))

          window               issued  loans  default_rate                                       used for
   fold 1: train Jan 2013 to Dec 2013   5349         0.159                            tuning and choosing
fold 1: validate Jan 2014 to Jun 2014   3997         0.167                            tuning and choosing
   fold 2: train Jan 2013 to Jun 2014   9346         0.163                            tuning and choosing
fold 2: validate Jul 2014 to Dec 2014   5027         0.191                            tuning and choosing
  final training Jan 2013 to Dec 2014  14373         0.172                       fitting the saved models
         2015 H1 Jan 2015 to Jun 2015   6712         0.200 checking the decision thresholds (notebook 09)
         2015 H2 Jul 2015 to Dec 2015   8915         0.201                 reported results (notebook 09)
    2016 holdout                 2016   5000           NaN           scored only, no labels (notebook 10)


**Reading it.** Each fold trains on the past and validates on the next six months, so a choice that only works because it has seen the future cannot win. A random split would mix 2014 loans into training and testing and flatter the model. **Every choice was made by a rule applied to the folds**: inputs, settings, the model family and the calibration. One decision was *prompted* by a 2015 result: the first model only matched the platform's grade on 2015, which led to revision 2 (section 8). The evidence for that revision and the rule that picked the final model came from the folds alone, and `DESIGN.md` section 11 says so.

One subtlety: the contract's WoE values were fitted on all loans up to 2014, labels included, so a fold's validation loans helped make them. For tuning only, WoE is **refitted inside each fold on that fold's training part** (`Trainer.__init__`); the contract itself is untouched.

## 3. One command, and what it guarantees

`uv run python -m credit_risk.model.train --config config/train.yaml` runs the whole thing. Every setting lives in the config file:

In [4]:
print(Path("config/train.yaml").read_text())

# Settings for training and evaluating the PD model. Everything the training run depends on is here.
seed: 42
data_dir: data
contract_dir: contract
model_dir: models
fit_cutoff: "2014-12-31"

# time-ordered tuning folds inside the training years (the future never trains the past)
folds:
  - {train_end: "2013-12-31", valid_start: "2014-01-01", valid_end: "2014-06-30"}
  - {train_end: "2014-06-30", valid_start: "2014-07-01", valid_end: "2014-12-31"}

# 2015 is the out-of-time validation year: thresholds are chosen on the first half, results reported on the second
policy_window: {start: "2015-01-01", end: "2015-06-30"}
report_window: {start: "2015-07-01", end: "2015-12-31"}

# decision policy assumptions (from the brief)
loss_given_default: 0.70      # a defaulted loan loses 70% of the amount
margin_if_repaid: 0.10        # a repaid loan earns 10% of the amount
min_approval_margin: 0.05     # auto-approve only where the expected margin is at least 5% of the loan

lightgbm:
  grid:
    num

In [5]:
print("transform:", type(tr.transformer).__name__, "| contract version", tr.transformer.contract_version, "| loaded from contract/transform.joblib and only ever asked to transform")
print("servicing columns (or the leaked recovery flag) in the modelling frame:", sorted(set(SERVICING_COLUMNS) & set(tr.frame.columns)) or "none")

# fail fast: the same entry point that builds the training frame refuses bad data instead of training on it
raw = load_applications(cfg["data_dir"]).head(300)
for what, field, bad in [("a required field missing (annual_inc)", "annual_inc", raw.drop(columns=["annual_inc"])),
                         ("utilisation sent as a fraction for the whole batch", "revol_util", raw.assign(revol_util=raw.revol_util / 100)),
                         ("a term the product does not offer (48 months)", "term", raw.assign(term=48))]:
    try:
        build_frame(tr.transformer, bad)
        print(f"\n{what}: ACCEPTED (unexpected)")
    except (ContractViolation, ValueError) as e:
        line = next(l.strip(" -") for l in str(e).splitlines() if field in l)  # the message lists every problem; show the one about this field
        print(f"\n{what}: refused -> {line[:160]}")

transform: ApplicationTransformer | contract version 1.0.0 | loaded from contract/transform.joblib and only ever asked to transform
servicing columns (or the leaked recovery flag) in the modelling frame: none

a required field missing (annual_inc): refused -> missing required field 'annual_inc'

utilisation sent as a fraction for the whole batch: refused -> ['revol_util'] look like fractions (0.91) but must be percentages (91); multiply the batch by 100 before scoring

a term the product does not offer (48 months): refused -> 'term': 300 value(s) not allowed (e.g. np.int64(48)); allowed: [36, 60]


**Reading it.** The run loads `transform.joblib` from the contract and calls only `transform` on it, never `fit`. The raw data is checked against `data_spec.json` before cleaning and the cleaned values again after; a batch that breaks the spec stops the run with a message rather than producing a model. Servicing columns are dropped by the contract and a guard in `build_frame` refuses to continue if any reached the frame (a test checks this as well). The seed (42) is set in the config and passed to every model.

## 4. The baseline: grade alone, and the WoE scorecard

Two references. **Grade alone** is the platform's own sub-grade (A1 to G5) turned into a PD by a one-input logistic regression: it is what we must beat. The **scorecard** is the interpretable baseline the brief asks for: logistic regression on the contract's WoE columns.

The scorecard was tuned over two settings: how strongly the weights are shrunk (`C`, smaller is stronger) and how many WoE columns to keep (`top_k`, the ones most related to default). Mean AUC over the two folds, for each pair:

In [6]:
sc = run[run.input_set == "woe_scorecard"].copy()
sc[["C", "top_k"]] = sc.params.apply(lambda s: pd.Series(json.loads(s)))
print(sc.pivot_table(index="top_k", columns="C", values="mean_auc").round(4).to_string())
g = run[run.input_set == "grade_alone"].iloc[0]
print(f"\ngrade alone on the same folds: {g.fold1_auc:.4f} and {g.fold2_auc:.4f} (mean {g.mean_auc:.4f})")
print(f"chosen scorecard: C = {sc.loc[sc.mean_auc.idxmax(), 'C']}, top_k = {int(sc.loc[sc.mean_auc.idxmax(), 'top_k'])}, mean AUC {sel['scorecard_mean_auc']:.4f}")

C        0.03    0.10    0.30    1.00    3.00
top_k                                        
8.0    0.6878  0.6886  0.6888  0.6887  0.6887
12.0   0.6917  0.6926  0.6928  0.6928  0.6928
16.0   0.6932  0.6941  0.6941  0.6940  0.6940
20.0   0.6952  0.6964  0.6966  0.6965  0.6963
33.0   0.6966  0.6983  0.6987  0.6981  0.6978

grade alone on the same folds: 0.6844 and 0.7103 (mean 0.6973)
chosen scorecard: C = 0.3, top_k = 33, mean AUC 0.6987


The tuning rule (in `tune_scorecard`): take the best mean AUC; any setting within 0.001 of it counts as a tie, and ties go to the simpler scorecard (fewer columns), then the lower Brier score.

### The sign rule, and the scorecard as points
WoE here is `ln(share of repaid / share of defaulted)`: a **higher WoE means a safer bin**. So every weight must be **zero or negative**, otherwise a safer bin would raise PD and its reason code would be nonsense. `LogitPD` enforces this: a column whose weight comes out positive is removed and the model refitted, until none remains.

In [7]:
card = joblib.load("models/woe_scorecard.joblib").model
w = pd.Series(card.model_.coef_[0], index=card.columns).sort_values()
print(f"{len(card.columns)} WoE columns kept; columns removed by the sign rule: {card.dropped_ or 'none'}; largest weight: {w.max():+.3f} (must be <= 0)")
print(w.round(3).to_string())

29 WoE columns kept; columns removed by the sign rule: ['revol_bal_log_woe', 'mort_acc_capped_woe', 'pub_rec_capped_woe', 'pub_rec_bankruptcies_capped_woe']; largest weight: -0.040 (must be <= 0)
mths_since_last_delinq_woe          -1.026
term_woe                            -0.994
emp_length_years_woe                -0.734
purpose_group_woe                   -0.711
delinq_2yrs_capped_woe              -0.700
inq_last_6mths_capped_woe           -0.698
total_rev_hi_lim_log_woe            -0.651
bc_util_treated_woe                 -0.634
revol_bal_to_income_treated_woe     -0.532
num_tl_op_past_12m_woe              -0.467
fico_mid_treated_woe                -0.461
home_ownership_group_woe            -0.455
open_acc_share_treated_woe          -0.412
recent_accounts_share_treated_woe   -0.394
acc_open_past_24mths_woe            -0.390
dti_treated_woe                     -0.388
installment_to_income_treated_woe   -0.365
open_acc_woe                        -0.307
annual_inc_log_woe            

A scorecard is usually shown as **points**. With the common convention that 20 points double the odds of repaying, a bin is worth `-weight x (bin WoE - average WoE) x 20 / ln 2` points, measured against the average applicant. Two fields as examples:

In [8]:
spec = tr.spec
pdo = 20 / np.log(2)
def points(field):
    col = f"{field}_woe"
    i = card.columns.index(col)
    entry = spec["woe"][field]
    rows = []
    for k, b in enumerate(entry["bins"]):
        if "edges" in entry:
            e = entry["edges"]
            fmt = lambda x: f"{x:,.0f}" if abs(x) >= 100 else f"{x:.2f}"
            lo = "-inf" if k == 0 else fmt(e[k - 1])
            hi = "inf" if k >= len(e) else fmt(e[k])
            what = f"{lo} to {hi}"
        else:
            what = b["label"]
        rows.append({"bin": what, "loans": b["loans"], "default_rate": b["default_rate"], "woe": b["woe"], "points": round(-card.model_.coef_[0][i] * (b["woe"] - card.mean_[i]) * pdo, 1)})
    return pd.DataFrame(rows)
for field in ["fico_mid_treated", "loan_to_income_treated"]:
    print(f"\n{field}:")
    print(points(field).to_string(index=False))


fico_mid_treated:
        bin  loans  default_rate     woe  points
-inf to 667   1281        0.2326 -0.3747    -5.5
 667 to 677   2679        0.2176 -0.2881    -4.4
 677 to 687   2307        0.1938 -0.1421    -2.4
 687 to 692   1040        0.1913 -0.1277    -2.2
 692 to 697   1052        0.1673  0.0355    -0.0
 697 to 707   1690        0.1521  0.1498     1.5
 707 to 722   1874        0.1505  0.1623     1.6
 722 to 732    803        0.1233  0.3903     4.7
 732 to 752    798        0.0890  0.7528     9.5
 752 to inf    849        0.0777  0.8995    11.5

loan_to_income_treated:
         bin  loans  default_rate     woe  points
-inf to 0.08   1415        0.1102  0.5179     1.7
0.08 to 0.12   1437        0.1169  0.4519     1.4
0.12 to 0.15   1459        0.1330  0.3052     0.9
0.15 to 0.22   3594        0.1503  0.1643     0.4
0.22 to 0.30   2837        0.1967 -0.1611    -0.7
0.30 to 0.43   2912        0.2332 -0.3776    -1.4
 0.43 to inf    719        0.2545 -0.4947    -1.8


**Reading it.** Points rise with FICO and fall as the loan grows relative to income, as a credit officer would expect, because each bin's default rate moves in one direction (the binning in notebook 04 guarantees it) and the weight has the right sign. On the folds the scorecard alone is level with grade alone (0.699 against 0.697): from application data only, it matches the platform's own score but does not beat it.

## 5. The stronger model: LightGBM with monotone constraints

Gradient boosting was tuned on the same folds over a small grid (tree size, minimum loans per leaf, number of trees, regularisation) with and without class weights: 32 trials per input set.

In [9]:
grid = cfg["lightgbm"]["grid"]
print("grid:", grid, "| class_weight:", cfg["lightgbm"]["class_weight"], "->", int(np.prod([len(v) for v in grid.values()])) * len(cfg["lightgbm"]["class_weight"]), "trials per input set")
print("\ndeclared monotone directions (+1: PD may only rise as the field rises; -1: only fall):")
d = pd.Series({c: v for c, v in CONSTRAINTS.items() if v != 0 and c in INPUT_SETS["both"]})
for v, name in [(1, "+1 (never lower PD as it rises)"), (-1, "-1 (never raise PD as it rises)")]:
    print(f"\n{name}:", textwrap.fill(", ".join(d[d == v].index), 160))

grid: {'num_leaves': [7, 15], 'min_child_samples': [50, 100], 'n_estimators': [150, 300], 'reg_lambda': [1.0, 10.0]} | class_weight: [None, 'balanced'] -> 32 trials per input set

declared monotone directions (+1: PD may only rise as the field rises; -1: only fall):

+1 (never lower PD as it rises): loan_amnt_capped, installment_capped, dti_treated, revol_util_treated, bc_util_treated, delinq_2yrs_capped, pub_rec_capped, pub_rec_bankruptcies_capped,
inq_last_6mths_capped, acc_open_past_24mths, num_tl_op_past_12m, loan_to_income_treated, installment_to_income_treated, revol_bal_to_income_treated,
recent_accounts_share_treated, factor_loan_size, factor_revolving_utilization, factor_recent_account_opening, factor_debt_burden, term

-1 (never raise PD as it rises): annual_inc_capped, avg_cur_bal_capped, total_rev_hi_lim_capped, tot_hi_cred_lim_capped, fico_mid_treated, credit_history_months_treated, factor_credit_capacity,
factor_revolving_credit_size


**Why these directions.** They are credit logic, not patterns found in the data: more debt for the income (`dti`, loan-to-income, installment-to-income), higher utilisation, more recent inquiries and new accounts, delinquencies and public records should **never lower** PD; a higher FICO score and a longer credit history should **never raise** it. Without a constraint, a tree can learn a small dip by chance (for example, PD falling between 30% and 32% utilisation), and an applicant could then lower their PD by borrowing more. The constraints rule that out, which also keeps reason codes consistent.

Each constraint was then **audited**: switch it off alone and see whether the folds improve. A constraint whose removal raises the mean AUC by more than 0.0005 is dropped, because the data disagree with it.

In [10]:
audit = pd.read_csv("reports/monotone_constraint_audit.csv").sort_values("auc_gain_if_constraint_removed", ascending=False)
print(audit.to_string(index=False))
print(f"\ndeclared {sel['monotone_declared']}, kept {len(sel['monotone_kept'])}; "
      f"AUC of the audited constraints minus no constraints at all: {-sel['monotone_cost_auc']:+.4f} (the limit allowed a cost of up to {cfg['lightgbm']['max_cost_of_a_monotone_constraint_auc']})")

                        field  direction  auc_gain_if_constraint_removed
            annual_inc_capped         -1                         0.00089
       tot_hi_cred_lim_capped         -1                         0.00060
installment_to_income_treated          1                         0.00059
             factor_loan_size          1                         0.00054
             fico_mid_treated         -1                         0.00051
           delinq_2yrs_capped          1                         0.00009
recent_accounts_share_treated          1                         0.00007
  pub_rec_bankruptcies_capped          1                         0.00000
               pub_rec_capped          1                         0.00000
credit_history_months_treated         -1                        -0.00005
             loan_amnt_capped          1                        -0.00013
      total_rev_hi_lim_capped         -1                        -0.00016
           factor_debt_burden          1           

**Reading it.** Most constraints cost nothing or help (a negative number means removing the constraint makes the folds worse). Five are dropped, each by a hair (0.0005 to 0.0009): income, total credit limit, installment-to-income, the loan-size factor and FICO. That does not mean a higher FICO raises risk; it means the trees use FICO together with other fields in ways a strict one-direction rule slightly blocks, and the scorecard member (where FICO's points rise steadily, section 4) still carries the expected direction. The audited set as a whole does **better** than no constraints at all (+0.004): the constraints act as a sensible prior that keeps the trees from fitting noise. Here the explainable choice is also the more accurate one.

## 6. Class imbalance

About 1 loan in 6 defaults in the training years (17%). The usual remedies are resampling or class weights that make defaults count more in training. Both distort the probabilities (a model trained as if defaults were common predicts too high a PD), and this model's output is used as a probability in a profit formula. So the tuning tried **no weighting against balanced weighting** and let the folds decide, judged by AUC with the Brier score as tie-break:

In [11]:
lg = run[(run.family == "lightgbm") & (run.monotone == "True") & (run.note.isna())].copy()
lg["class_weight"] = lg.class_weight.fillna("none")
best = lg.loc[lg.groupby(["input_set", "class_weight"]).mean_auc.idxmax()].set_index(["input_set", "class_weight"])[["mean_auc", "mean_brier"]]
print(best.round(4).rename(columns={"mean_auc": "best mean AUC", "mean_brier": "Brier of that setting"}).to_string())
print("\nchosen class weight for every boosted model:", {n: joblib.load(f"models/{n}.joblib").model.class_weight for n in ["lightgbm", "lightgbm_plus_grade"]})

                                best mean AUC  Brier of that setting
input_set         class_weight                                      
both              balanced             0.7006                 0.2160
                  none                 0.6997                 0.1356
factors           balanced             0.6971                 0.2195
                  none                 0.6969                 0.1358
raw               balanced             0.6979                 0.2166
                  none                 0.6971                 0.1362
raw_plus_subgrade balanced             0.7110                 0.2143
                  none                 0.7116                 0.1344



chosen class weight for every boosted model: {'lightgbm': None, 'lightgbm_plus_grade': None}


**Reading it.** Balanced weights change the ranking by at most 0.001 AUC either way, which the tuning treats as a tie, and they make the probabilities much worse (Brier about 0.21 against 0.14: the weighted model predicts far too many defaults). The tie goes to the lower Brier score, so **no weighting** is used: the model trains on the real mix and is calibrated afterwards (section 9). The imbalance is mild, and with about 14,000 training loans there are about 2,500 defaults to learn from, so nothing needs to be invented by resampling.

## 7. The ablation: raw fields vs factor scores vs both

The brief asks which of the data scientist's outputs actually help. Each model family was tuned on three input sets: the treated raw fields, the factor scores (plus the categories and term), and both.

In [12]:
ab = pd.DataFrame([{"family": "gradient boosting", "inputs": k, "mean AUC (time folds)": v} for k, v in sel["ablation_lightgbm"].items()] +
                  [{"family": "logistic", "inputs": {"woe_only": "WoE (raw fields)", "factors_plus_category_woe": "factor scores + category WoE", "woe_plus_factors": "WoE + factor scores"}[k], "mean AUC (time folds)": v} for k, v in sel["ablation_logistic"].items()])
print(ab.round(4).to_string(index=False))
lb = sel["ablation_lightgbm"]
print(f"\nrule: a larger input set must add at least {MIN_GAIN_TO_ADD_FACTORS} AUC; both minus raw = {lb['both'] - lb['raw']:+.4f} -> boosted model uses '{sel['chosen_input_set']}'")

           family                       inputs  mean AUC (time folds)
gradient boosting                          raw                 0.6970
gradient boosting                      factors                 0.6967
gradient boosting                         both                 0.6997
         logistic             WoE (raw fields)                 0.6987
         logistic factor scores + category WoE                 0.6975
         logistic          WoE + factor scores                 0.6952

rule: a larger input set must add at least 0.002 AUC; both minus raw = +0.0027 -> boosted model uses 'both'


**Reading it.** Factor scores **alone** do about as well as the raw fields in both families: eight factors carry nearly all the information of some 25 fields, which supports the factor analysis. **Adding** them to the raw fields helps the boosted model a little (+0.003, just above the rule) and does not help the scorecard (it scores slightly lower). This is the feedback for the data scientist: the factors are a faithful compression, but there is little new signal in them for a model that already sees the fields. The model card records this.

## 8. The platform's sub-grade, and the selection rule

`DESIGN.md` first planned to keep the platform's grade and interest rate out of the model and use them only as a benchmark. During development it became clear that application data alone **matches** the grade but does not beat it (section 4), while the sub-grade added to our inputs beats it clearly. `DESIGN.md` section 11 records the change: **the sub-grade is allowed as an input** (it is known at application time; the grade letter and the rate are coarser and near-copies of it), its weight must be positive, and the independent scorecard stays available as a fallback (`platform_score: none` in the config).

The rule for choosing among the candidates was written down **before** the comparison:
- the boosted model replaces the scorecard only if it is ahead by at least **0.005** AUC on average **and ahead in every fold**;
- a later change (the average of the two, from the screen in `DESIGN.md` section 12) must add at least **0.003** on average **and be ahead in every fold**.

In [13]:
# the fold AUCs of the configuration each tuning step actually chose (ties go to the simpler or better-calibrated setting), as stored in each saved model
names = {"grade_baseline": "grade alone", "woe_scorecard": "scorecard (no platform score)", "lightgbm": "boosting (no platform score)",
         "scorecard_plus_grade": "scorecard + sub-grade", "lightgbm_plus_grade": "boosting + sub-grade", "scorecard_boosted_average": "average of the two (log-odds)"}
t = pd.DataFrame({label: joblib.load(f"models/{n}.joblib").metadata["fold_auc"] for n, label in names.items()}, index=["fold1_auc", "fold2_auc"]).T
t["mean_auc"] = t.mean(axis=1)
print(t.round(4).to_string())

scg, lgg, avg = (t.loc[k, ["fold1_auc", "fold2_auc"]].to_numpy() for k in ["scorecard + sub-grade", "boosting + sub-grade", "average of the two (log-odds)"])
wins, gains = boosted_model_wins(lgg, scg)
print(f"\nboosting + sub-grade vs scorecard + sub-grade: gain per fold {np.round(gains, 4)}, mean {np.mean(gains):+.4f} -> replaces it: {wins} (needs +{MIN_GAIN_FOR_BOOSTING} and ahead in every fold)")
wins2, gains2 = boosted_model_wins(avg, scg, AVERAGE_MIN_GAIN)
print(f"average vs scorecard + sub-grade:           gain per fold {np.round(gains2, 4)}, mean {np.mean(gains2):+.4f} -> replaces it: {wins2} (needs +{AVERAGE_MIN_GAIN} and ahead in every fold)")
print("selected by the training run:", sel["selected"])

                               fold1_auc  fold2_auc  mean_auc
grade alone                       0.6844     0.7102    0.6973
scorecard (no platform score)     0.6861     0.7112    0.6986
boosting (no platform score)      0.6891     0.7142    0.7016
scorecard + sub-grade             0.6983     0.7259    0.7121
boosting + sub-grade              0.7015     0.7217    0.7116
average of the two (log-odds)     0.7039     0.7276    0.7158

boosting + sub-grade vs scorecard + sub-grade: gain per fold [ 0.0032 -0.0042], mean -0.0005 -> replaces it: False (needs +0.005 and ahead in every fold)
average vs scorecard + sub-grade:           gain per fold [0.0056 0.0017], mean +0.0036 -> replaces it: True (needs +0.003 and ahead in every fold)
selected by the training run: scorecard_boosted_average


**Reading it.** On its own, boosting with the sub-grade is better in one fold and worse in the other, so it does **not** replace the scorecard. Their **average** (taken in log-odds, so it is still a probability model) is ahead of the scorecard in both folds and by more than the threshold on average: two different kinds of model make different mistakes, and averaging cancels some of them. It is the selected model. Each member stays explainable, and the reason codes of the average are exact (notebook 10).

These numbers come from the experiment log. To show they are reproducible, here is the selected model re-evaluated from scratch on the same folds, with the settings stored in the saved model:

In [14]:
saved = joblib.load("models/pd_model.joblib").model
logit, boost = saved.members
factory = lambda: LogOddsAverage([LogitPD(SCORECARD_COLUMNS + PLATFORM_COLUMNS, logit.C, set(PLATFORM_COLUMNS), cfg["seed"], nonpositive=True, top_k=logit.top_k, positive=PLATFORM_COLUMNS),
                                  LgbmPD(boost.columns, boost.params, boost.class_weight, boost.monotone, cfg["seed"])], saved.field_of)
aucs, briers, _ = tr.evaluate(factory, woe_fold=True)
print("re-evaluated fold AUCs:", np.round(aucs, 4), "| recorded at training:", meta["fold_auc"])

re-evaluated fold AUCs: [0.7039 0.7276] | recorded at training: [0.7039, 0.7276]


## 9. Calibration

A ranking is not yet a probability. The PD feeds the profit formula, so it must mean what it says. Each model's scores on the validation folds (scores for loans the model had not seen) were used to fit a calibrator, **separately for 36- and 60-month loans** (they behave like different products, notebook 01). Two methods were compared, each fitted on one fold and scored on the other:
- **Platt scaling:** a logistic curve on the model's log-odds; two parameters, smooth.
- **Isotonic regression:** any increasing step function; more flexible, can overfit.

In [15]:
cc = pd.DataFrame(meta["calibration_brier_cross_check"]).T
cc.index = [f"{t}-month loans" for t in cc.index]
cc["chosen"] = [meta["calibration"][t] for t in meta["calibration"]]
print(cc.round(5).to_string())

                  platt  isotonic chosen
36-month loans  0.11047   0.11048  platt
60-month loans  0.19527   0.19619  platt


**Reading it.** Platt scaling has the lower cross-checked Brier score for both terms (for 36-month loans the two are practically tied, and Platt is the simpler one), so it is used for both. Notebook 09 checks the result on 2015.

## 10. Lineage: what is saved with the model

`models/metadata.json` travels with `models/pd_model.joblib`. The brief asks for the git commit, a training-data hash, the training date, the contract version and the thresholds:

In [16]:
keys = ["model", "contract_version", "fit_cutoff", "training_loans", "training_years", "git_commit", "git_dirty", "training_data_hash", "training_date", "seed", "thresholds", "calibration", "fold_auc", "reference_group"]
for k in keys:
    print(f"{k:<20} {meta[k]}")
print("\nthe commit exists in this repository:", subprocess.run(["git", "cat-file", "-e", meta["git_commit"]]).returncode == 0)
print("the contract version matches the transform:", meta["contract_version"] == tr.transformer.contract_version)
print("the training-data hash is reproduced from the contract:", data_hash(tr.train, []) == meta["training_data_hash"])

model                scorecard_boosted_average
contract_version     1.0.0
fit_cutoff           2014-12-31
training_loans       14373
training_years       ['2013-01-01', '2014-12-01']
git_commit           4d83cf8ef66e5ea71c64fccac89aa5f2d256581c
git_dirty            False
training_data_hash   4ff160121523e1936f71e23824207c51c881b5e92af3db7e8e5385c85708fc44
training_date        2026-11-02
seed                 42
thresholds           {'approve_below': 0.0625, 'decline_at_or_above': 0.125}
calibration          {'36': 'platt', '60': 'platt'}
fold_auc             [0.7039, 0.7276]
reference_group      2,181 training loans the model approves

the commit exists in this repository: True
the contract version matches the transform: True
the training-data hash is reproduced from the contract: True


**Reading it.** The model was trained from a clean checkout (`git_dirty` false), so the commit identifies exactly the code that produced it. The contract version is checked again at load time: `predict.py` refuses a model built on a different contract (notebook 10). For models whose inputs are WoE columns, the data fingerprint covers the training loans' ids and labels; the input values themselves are fixed by the versioned contract. The thresholds come from the profit assumptions, not from the data (notebook 09).

## Summary: what the brief asked, and where it is answered

| Brief item | Answered in |
|---|---|
| `DESIGN.md` committed before model code, updated at the end | section 1 (sections 10 to 12 of `DESIGN.md` record the changes) |
| Time split around the fit cutoff; time-aware tuning | section 2 |
| One command, contract transform unchanged, fail-fast validation, config, seeds, experiment log, no servicing data | section 3 (`reports/experiments.csv`) |
| Scorecard on the contract's WoE bins | section 4 |
| A stronger model with justified monotone constraints | section 5 |
| Class imbalance | section 6 |
| Ablation: raw vs factors vs both, and what helped | section 7 (and `model_card.md`) |
| Grade and interest rate as inputs, and why | section 8 (`DESIGN.md` sections 3 and 11) |
| Calibrated probabilities | section 9 (checked on 2015 in notebook 09) |
| Metadata: commit, data hash, date, contract version, thresholds | section 10 |